# Notebook 06 — Swin-Tiny retrain (recovery run)## Why this notebook existsThe shipped `swin` checkpoint does **not** reproduce its own stored metrics. Measuredon the real test split, in two independent environments (Kaggle T4 and a local CPU),agreeing to four decimal places with 100% prediction agreement:| | stored in checkpoint | reproduced ||---|---|---|| test accuracy | 85.39% | see `paper/FACT_CHECK.md` || train accuracy | 92.50% (history) | **62.00%** || melanoma sensitivity | 98.59% | 28.3% (on train) |A model fitted to 92.5% on its training images cannot later score 62% on those sameimages. The `model_state` in the file is therefore **not** the model that produced therecorded history or the reported test accuracy — most plausibly it was overwritten bya re-executed cell in the original Colab session, while `history` and `test_acc` werecarried over from the earlier, correct evaluation.The training *recipe* is intact, so this notebook re-runs it to recover usable weights.## What is identical to the originalModel class, head structure, train/val transforms, class-weighted loss with theclinical boosts, `WeightedRandomSampler`, AdamW with layer-wise LR decay, thewarmup+cosine `LambdaLR` schedule, AMP, gradient clipping at 1.0, the compositecheckpoint criterion, `SEED = 42`, and every value in `CFG`:| | value ||---|---|| lr | 1e-4 || weight decay | 3e-4 || dropout | 0.4 || drop_path_rate | 0.2 || label smoothing | 0.15 || batch size | 8 || warmup epochs | 5 || freeze epochs | 8 || patience | 5 || num_epochs | 30 |## What differs, and why1. **Kaggle paths** — `google.colab.drive` does not exist here.2. **`history.csv` written every epoch** — so the training record survives even if   cell outputs are lost, which is how the original provenance disappeared.3. **`timing.json`** — real wall-clock and GPU name, replacing the paper's   unsupported "≈6 h on 2×T4".4. **Save-then-verify** — the saved checkpoint is reloaded from disk and re-scored,   and the notebook asserts the reloaded model matches the in-memory one tensor by   tensor and reproduces the same test accuracy. **Had this existed originally, the   overwrite would have been caught immediately.**5. **MCC is computed and stored** — the paper quotes MCC values that appear in no   original checkpoint.## Expected trajectoryIf the recipe reproduces, expect roughly: epochs 1–8 frozenbackbone (val climbing to ~65–75%), unfreeze at epoch 9,then val ≈ 88.35% and test ≈ 85.39% by epoch 27.`cudnn.benchmark = True` and non-deterministic kernels mean the result will be *near*the original, not identical. Whatever this run produces becomes the paper's number.**Runtime: ~3.5-4.5 h on one T4.** Use **Save Version → Save & Run All (Commit)**,never an interactive session — a browser disconnect is how provenance gets lost.

## 1. Setup — paths, seed, device

In [ ]:
# ============================ EDIT THESE ============================DATA_ROOT = '/kaggle/input/datasets/jainmanav20/ipd-skin-dataset/IPD_Datasets_Final_Split'OUT_DIR   = '/kaggle/working'# ====================================================================!pip install -q timmimport os, random, warnings, json, time, shutilfrom pathlib import Pathfrom collections import Counterimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimfrom torch.utils.data import DataLoader, WeightedRandomSamplerfrom torchvision import transformsfrom torchvision.datasets import ImageFolderimport timmfrom sklearn.metrics import (    accuracy_score, classification_report, roc_auc_score, confusion_matrix,    matthews_corrcoef, balanced_accuracy_score, f1_score,)from sklearn.preprocessing import label_binarizewarnings.filterwarnings('ignore')# Seed 42 — the value the original run actually used (the paper's "seed 32" is wrong).SEED = 42random.seed(SEED); np.random.seed(SEED)torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'print(f'Device : {DEVICE}')print(f'GPU    : {GPU_NAME}')if DEVICE.type == 'cuda':    props = torch.cuda.get_device_properties(0)    print(f'VRAM   : {props.total_memory/1024**3:.1f} GB')    # Identical to the original notebook: benchmark on, determinism NOT forced.    torch.backends.cuda.matmul.allow_tf32 = True    torch.backends.cudnn.allow_tf32 = True    torch.backends.cudnn.benchmark = Trueprint(f'torch {torch.__version__} | timm {timm.__version__}')LOCAL_ROOT = Path(DATA_ROOT)SAVE_DIR = Path(OUT_DIR)SAVE_DIR.mkdir(parents=True, exist_ok=True)for split in ['train', 'val', 'test']:    d = LOCAL_ROOT/split    assert d.exists(), f'{split} folder missing at {d}'print('data root OK')

## 2. Config — verbatim from the original notebookEvery value below is copied from `src/models/swin*_5class.ipynb`. `ckpt_path` isrepointed at `/kaggle/working` (the original wrote to `/content`).

In [ ]:
CFG = dict(    model_name      = 'swin_tiny_patch4_window7_224',    img_size        = 224,    batch_size      = 8,    num_epochs      = 30,    lr              = 1e-4,    weight_decay    = 3e-4,    dropout         = 0.4,    drop_path_rate  = 0.2,    label_smooth    = 0.15,    warmup_epochs   = 5,    use_amp         = True,    num_workers     = 2,       # Kaggle gives more than Colab's 1 core    freeze_epochs   = 8,    patience        = 5,    lr_decay_rate   = 0.1,    melanoma_boost  = 2.0,    bcc_boost       = 2.0,    tinea_boost     = 3.0,    eczema_boost    = 1.5,    ckpt_path       = str(SAVE_DIR/'best_swin_tiny_5cls_v2.pth'),)for k, v in CFG.items():    print(f'  {k:16s} {v}')

## 3. Transforms — verbatim (train augmentation + eval resize only)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]IMAGENET_STD  = [0.229, 0.224, 0.225]train_tf = transforms.Compose([    transforms.Resize((CFG['img_size'] + 32, CFG['img_size'] + 32)),    transforms.RandomCrop(CFG['img_size']),    transforms.RandomHorizontalFlip(),    transforms.RandomVerticalFlip(),    transforms.RandomRotation(30),    transforms.ColorJitter(        brightness=0.4, contrast=0.4,        saturation=0.3, hue=0.1    ),    transforms.RandomAffine(        degrees=0, translate=(0.1, 0.1),        shear=10, scale=(0.85, 1.15)    ),    transforms.RandomGrayscale(p=0.05),    transforms.ToTensor(),    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3)),])val_tf = transforms.Compose([    transforms.Resize((CFG['img_size'], CFG['img_size'])),    transforms.ToTensor(),    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),])print('Transforms ready.')print('NOTE: no CLAHE and no Gaussian/median denoising — neither exists in any')print('      original code path, and the paper should not claim them.')

## 4. Datasets and loaders — verbatimRestricted to the five paper classes. The corpus may also contain Acne,Lichen_Planus, Psoriasis and Vitiligo; those folders are ignored.

In [ ]:
PAPER_CLASSES = ['Basal_Cell_Carcinoma', 'Eczema', 'Keratosis', 'Melanoma', 'Tinea']class FiveClassImageFolder(ImageFolder):    # class_to_idx pinned to the alphabetical paper order, so index 0-4 means the    # same thing here as in every existing checkpoint.    def find_classes(self, directory):        return list(PAPER_CLASSES), {c: i for i, c in enumerate(PAPER_CLASSES)}TRAIN_DIR = LOCAL_ROOT/'train'VAL_DIR   = LOCAL_ROOT/'val'TEST_DIR  = LOCAL_ROOT/'test'train_ds = FiveClassImageFolder(TRAIN_DIR, transform=train_tf)val_ds   = FiveClassImageFolder(VAL_DIR,   transform=val_tf)test_ds  = FiveClassImageFolder(TEST_DIR,  transform=val_tf)CLASSES = train_ds.classesNUM_CLS = len(CLASSES)assert CLASSES == PAPER_CLASSES, CLASSESprint(f'Classes ({NUM_CLS}): {CLASSES}')print(f'Train : {len(train_ds)} | Val : {len(val_ds)} | Test : {len(test_ds)}')class_counts = Counter(train_ds.targets)print('\nTraining class distribution:')for i, cls in enumerate(CLASSES):    print(f'  {cls:<25} {class_counts[i]:>5}')class_weights_sampler = 1.0 / torch.tensor(    [class_counts[i] for i in range(NUM_CLS)], dtype=torch.float)sample_w = [class_weights_sampler[l] for l in train_ds.targets]sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler,                          num_workers=CFG['num_workers'], pin_memory=True)val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,                          num_workers=CFG['num_workers'], pin_memory=True)test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,                          num_workers=CFG['num_workers'], pin_memory=True)print('DataLoaders ready.')SPLIT_SIZES = dict(train=len(train_ds), val=len(val_ds), test=len(test_ds))TRAIN_DIST = {c: int(class_counts[i]) for i, c in enumerate(CLASSES)}

## 5. Model — `SwinSkinClassifier`, verbatim

In [ ]:
class SwinSkinClassifier(nn.Module):    def __init__(self, model_name, num_classes, dropout=0.4,                 drop_path_rate=0.2, pretrained=True):        super().__init__()        self.backbone = timm.create_model(            model_name,            pretrained     = pretrained,            num_classes    = 0,            drop_rate      = dropout,            drop_path_rate = drop_path_rate,        )        feat_dim = self.backbone.num_features  # 768        self.classifier = nn.Sequential(            nn.LayerNorm(feat_dim),            nn.Dropout(dropout),            nn.Linear(feat_dim, 256),            nn.GELU(),            nn.Dropout(dropout / 2),            nn.Linear(256, num_classes),        )        print(f'Swin-Tiny | features={feat_dim} | '              f'classes={num_classes} | dropout={dropout} | '              f'drop_path={drop_path_rate}')    def forward(self, x): return self.classifier(self.backbone(x))    def freeze_backbone(self):        for p in self.backbone.parameters(): p.requires_grad = False        n = sum(p.numel() for p in self.parameters() if p.requires_grad)        print(f'  Backbone frozen    | trainable: {n:,}')    def unfreeze_backbone(self):        for p in self.backbone.parameters(): p.requires_grad = True        n = sum(p.numel() for p in self.parameters() if p.requires_grad)        print(f'  Backbone unfrozen  | trainable: {n:,}')model = SwinSkinClassifier(    model_name     = CFG['model_name'],    num_classes    = NUM_CLS,    dropout        = CFG['dropout'],    drop_path_rate = CFG['drop_path_rate'],    pretrained     = True).to(DEVICE)model = model.to(memory_format=torch.channels_last)total     = sum(p.numel() for p in model.parameters())trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f'Total params    : {total:,}')print(f'Trainable params: {trainable:,}')# The head goes straight to 5 classes. There is no fusion network, no 512-d# projection and no 1536-d concatenation anywhere in this model — the paper's# Sec. 5.1-5.3 and Fig. 2 describe an architecture that was never built.assert isinstance(model.classifier[-1], nn.Linear)assert model.classifier[-1].out_features == NUM_CLS

## 6. Loss, optimiser and scheduler — verbatimClass weights are `1/sqrt(count)`, renormalised, then multiplied by the clinicalboosts (Melanoma ×2, BCC ×2, Tinea ×3, Eczema ×1.5) and renormalised again. Theschedule is linear warmup followed by cosine decay — **the paper omits both theschedule and the freeze phase.**

In [ ]:
counts_arr   = np.array([class_counts[i] for i in range(NUM_CLS)], dtype=np.float32)loss_weights = 1.0 / np.sqrt(counts_arr)loss_weights /= loss_weights.sum()melanoma_idx = CLASSES.index('Melanoma')bcc_idx      = CLASSES.index('Basal_Cell_Carcinoma')tinea_idx    = CLASSES.index('Tinea')eczema_idx   = CLASSES.index('Eczema')loss_weights[melanoma_idx] *= CFG['melanoma_boost']loss_weights[bcc_idx]      *= CFG['bcc_boost']loss_weights[tinea_idx]    *= CFG['tinea_boost']loss_weights[eczema_idx]   *= CFG['eczema_boost']loss_weights /= loss_weights.sum()weights_tensor = torch.tensor(loss_weights, dtype=torch.float32).to(DEVICE)criterion = nn.CrossEntropyLoss(    weight          = weights_tensor,    label_smoothing = CFG['label_smooth'])print('Class weights for loss:')for i, cls in enumerate(CLASSES):    print(f'  {cls:<25} {loss_weights[i]:.4f}')LOSS_WEIGHTS = {c: float(loss_weights[i]) for i, c in enumerate(CLASSES)}backbone_params = [p for n, p in model.named_parameters()                   if 'classifier' not in n and p.requires_grad]head_params     = [p for n, p in model.named_parameters()                   if 'classifier' in n and p.requires_grad]optimizer = optim.AdamW([    {'params': backbone_params, 'lr': CFG['lr'] * CFG['lr_decay_rate']},    {'params': head_params,     'lr': CFG['lr']},], weight_decay=CFG['weight_decay'])print(f'\nLayer-wise LR: backbone={CFG["lr"]*CFG["lr_decay_rate"]:.2e} | head={CFG["lr"]:.2e}')def lr_lambda(epoch):    if epoch < CFG['warmup_epochs']:        return (epoch + 1) / CFG['warmup_epochs']    progress = (epoch - CFG['warmup_epochs']) / max(1, CFG['num_epochs'] - CFG['warmup_epochs'])    return 0.5 * (1 + np.cos(np.pi * progress))scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)scaler    = torch.cuda.amp.GradScaler(enabled=CFG['use_amp'])print('Optimiser / Scheduler ready.')

## 7. Training loop — verbatim, plus per-epoch `history.csv`The loop body is unchanged from the original. The only addition is that `history`is flushed to `history.csv` after **every** epoch, so the training record exists ondisk independently of this notebook's cell outputs.Checkpoint selection uses the original composite criterion, computed on**validation only**:```composite = 0.4 x melanoma_sensitivity + 0.3 x bcc_sensitivity + 0.3 x val_acc```Expect ~3.5-4.5 h. Rows marked `*` saved a new best.

In [ ]:
def train_epoch(loader):    model.train()    total_loss, correct, total = 0.0, 0, 0    for imgs, lbls in loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        lbls = lbls.to(DEVICE, non_blocking=True)        with torch.cuda.amp.autocast(enabled=CFG['use_amp']):            logits = model(imgs)            loss   = criterion(logits, lbls)        optimizer.zero_grad(set_to_none=True)        scaler.scale(loss).backward()        scaler.unscale_(optimizer)        nn.utils.clip_grad_norm_(model.parameters(), 1.0)        scaler.step(optimizer)        scaler.update()        total_loss += loss.item() * imgs.size(0)        correct    += (logits.argmax(1) == lbls).sum().item()        total      += imgs.size(0)    return total_loss / total, correct / total@torch.no_grad()def val_epoch(loader):    model.eval()    total_loss, correct, total = 0.0, 0, 0    all_p, all_l = [], []    for imgs, lbls in loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        lbls = lbls.to(DEVICE, non_blocking=True)        with torch.cuda.amp.autocast(enabled=CFG['use_amp']):            logits = model(imgs)            loss   = criterion(logits, lbls)        total_loss += loss.item() * imgs.size(0)        correct    += (logits.argmax(1) == lbls).sum().item()        total      += imgs.size(0)        all_p.extend(logits.argmax(1).cpu().numpy())        all_l.extend(lbls.cpu().numpy())    all_p = np.array(all_p); all_l = np.array(all_l)    mel_tp = ((all_p == melanoma_idx) & (all_l == melanoma_idx)).sum()    mel_fn = ((all_p != melanoma_idx) & (all_l == melanoma_idx)).sum()    bcc_tp = ((all_p == bcc_idx) & (all_l == bcc_idx)).sum()    bcc_fn = ((all_p != bcc_idx) & (all_l == bcc_idx)).sum()    mel_sens = mel_tp / (mel_tp + mel_fn) if (mel_tp + mel_fn) > 0 else 0.0    bcc_sens = bcc_tp / (bcc_tp + bcc_fn) if (bcc_tp + bcc_fn) > 0 else 0.0    return total_loss / total, correct / total, mel_sens, bcc_sensprint('Sanity check...')imgs_s, lbls_s = next(iter(train_loader))print(f'Batch: {imgs_s.shape} | Labels: {lbls_s.shape}')del imgs_s, lbls_shistory = {    'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],    'gap': [], 'melanoma_sensitivity': [], 'bcc_sensitivity': [], 'composite': [],}best_composite, patience_cnt = 0.0, 0# Recorded at the SAVED epoch, not as a max() over all epochs. The original# notebook stored max(...) for these, which is why its best_val_acc came from a# different epoch than its best_composite.saved_epoch, saved_val_acc, saved_mel_sens, saved_bcc_sens = 0, 0.0, 0.0, 0.0t_start = time.time()epoch_seconds = []print(f'\n{"Ep":>4} {"TrLoss":>8} {"VlLoss":>8} {"TrAcc":>7} {"VlAcc":>7} '      f'{"MelSens":>8} {"BCCSens":>8} {"Comp":>7} {"LR":>9} {"min":>6}')print('-' * 92)for epoch in range(1, CFG['num_epochs'] + 1):    t_ep = time.time()    if epoch == 1:        model.freeze_backbone()    elif epoch == CFG['freeze_epochs'] + 1:        model.unfreeze_backbone()        backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]        head_params     = [p for n, p in model.named_parameters() if 'classifier' in n]        optimizer = optim.AdamW([            {'params': backbone_params, 'lr': CFG['lr'] * CFG['lr_decay_rate']},            {'params': head_params,     'lr': CFG['lr']},        ], weight_decay=CFG['weight_decay'])        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)        print('  -> Full fine-tuning with layer-wise LR.')    tr_loss, tr_acc               = train_epoch(train_loader)    vl_loss, vl_acc, mel_s, bcc_s = val_epoch(val_loader)    scheduler.step()    composite = 0.4 * mel_s + 0.3 * bcc_s + 0.3 * vl_acc    gap = tr_acc - vl_acc    history['train_loss'].append(tr_loss);         history['val_loss'].append(vl_loss)    history['train_acc'].append(tr_acc);           history['val_acc'].append(vl_acc)    history['gap'].append(gap)    history['melanoma_sensitivity'].append(mel_s); history['bcc_sensitivity'].append(bcc_s)    history['composite'].append(composite)    tag = ''    if composite > best_composite:        best_composite = composite        torch.save(model.state_dict(), CFG['ckpt_path'])        patience_cnt = 0; tag = ' *'        saved_epoch, saved_val_acc = epoch, vl_acc        saved_mel_sens, saved_bcc_sens = float(mel_s), float(bcc_s)    else:        patience_cnt += 1    el = time.time() - t_ep    epoch_seconds.append(el)    current_lr = optimizer.param_groups[-1]['lr']    gap_flag   = ' !' if gap > 0.10 else (' ~' if gap > 0.05 else '')    print(f'{epoch:>4} {tr_loss:>8.4f} {vl_loss:>8.4f} '          f'{tr_acc*100:>6.2f}% {vl_acc*100:>6.2f}% '          f'{mel_s*100:>7.2f}% {bcc_s*100:>7.2f}% '          f'{composite:>7.4f} {current_lr:>9.2e} {el/60:>6.1f}{gap_flag}{tag}')    # Flush every epoch: the record must not depend on cell outputs surviving.    hdf = pd.DataFrame(history)    hdf.insert(0, 'epoch', range(1, len(hdf) + 1))    hdf['epoch_seconds'] = epoch_seconds    hdf.to_csv(SAVE_DIR/'history.csv', index=False)    if patience_cnt >= CFG['patience']:        print(f'\nEarly stopping at epoch {epoch}.')        breaktrain_wall = time.time() - t_startepochs_run = len(history['train_loss'])print(f'\nBest composite    : {best_composite:.4f}  (epoch {saved_epoch})')print(f'Saved-epoch val   : {saved_val_acc*100:.2f}%')print(f'Saved-epoch melsens: {saved_mel_sens*100:.2f}%  (target >= 95%)')print(f'Epochs run        : {epochs_run} / {CFG["num_epochs"]}')print(f'Wall-clock        : {train_wall/3600:.2f} h  ({train_wall/60/epochs_run:.1f} min/epoch)')print(f'Avg gap (last 5)  : {np.mean(history["gap"][-5:])*100:.1f}%  (target < 5%)')

## 8. Evaluation on the held-out test splitThe best-composite checkpoint is reloaded from disk and scored once. MCC is computedhere because the paper quotes MCC values that exist in no original checkpoint.

In [ ]:
model.load_state_dict(torch.load(CFG['ckpt_path'], map_location=DEVICE))model.eval()all_preds, all_labels, all_probs = [], [], []with torch.no_grad():    for imgs, lbls in test_loader:        imgs   = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        logits = model(imgs)        all_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())        all_preds.extend(logits.argmax(1).cpu().numpy())        all_labels.extend(lbls.numpy())all_preds  = np.array(all_preds)all_labels = np.array(all_labels)all_probs  = np.array(all_probs)test_acc  = accuracy_score(all_labels, all_preds)y_bin     = label_binarize(all_labels, classes=list(range(NUM_CLS)))macro_auc = roc_auc_score(y_bin, all_probs, multi_class='ovr', average='macro')mcc       = matthews_corrcoef(all_labels, all_preds)bal_acc   = balanced_accuracy_score(all_labels, all_preds)macro_f1  = f1_score(all_labels, all_preds, average='macro')print(f'Test Accuracy    : {test_acc*100:.2f}%')print(f'Macro AUC        : {macro_auc:.4f}')print(f'MCC              : {mcc:.4f}')print(f'Balanced accuracy: {bal_acc*100:.2f}%')print(f'Macro F1         : {macro_f1:.4f}')print()print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))cm = confusion_matrix(all_labels, all_preds)PER_CLASS = {}for i, c in enumerate(CLASSES):    tp = int(cm[i, i]); fn = int(cm[i].sum() - tp)    fp = int(cm[:, i].sum() - tp); tn = int(cm.sum() - tp - fn - fp)    PER_CLASS[c] = dict(        support=int(cm[i].sum()),        recall=tp/(tp+fn) if tp+fn else 0.0,        precision=tp/(tp+fp) if tp+fp else 0.0,        specificity=tn/(tn+fp) if tn+fp else 0.0,        auc=float(roc_auc_score((all_labels == i).astype(int), all_probs[:, i])),    )    print(f'{c:24s} recall={PER_CLASS[c]["recall"]*100:6.2f}%  '          f'prec={PER_CLASS[c]["precision"]*100:6.2f}%  '          f'spec={PER_CLASS[c]["specificity"]*100:6.2f}%  '          f'AUC={PER_CLASS[c]["auc"]:.4f}')pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(SAVE_DIR/'confusion_test.csv')pd.DataFrame(PER_CLASS).T.to_csv(SAVE_DIR/'per_class_test.csv')

## 9. Save the checkpointSame wrapped format as the originals so `SkinEnsemble` and Notebook 1 load itunchanged, plus three additions: `mcc`, `saved_epoch`, and `metrics_at_saved_epoch`.The original notebook stored `best_val_acc = max(history['val_acc'])` and`best_melanoma_sens = max(history['melanoma_sensitivity'])` — maxima over *all*epochs, which need not come from the saved epoch. Those keys are kept forcompatibility, but the saved epoch's own values are recorded alongside them.

In [ ]:
save_path = SAVE_DIR/'swin_tiny_5cls_skin_v2.pth'payload = dict(    model_state        = model.state_dict(),    classes            = CLASSES,    config             = CFG,    best_composite     = float(best_composite),    # kept for compatibility with the original format (max over all epochs)    best_val_acc       = float(max(history['val_acc'])),    best_melanoma_sens = float(max(history['melanoma_sensitivity'])),    test_acc           = float(test_acc),    macro_auc          = float(macro_auc),    history            = history,    # --- additions ---    mcc                = float(mcc),    balanced_accuracy  = float(bal_acc),    macro_f1           = float(macro_f1),    per_class          = PER_CLASS,    saved_epoch        = int(saved_epoch),    epochs_run         = int(epochs_run),    metrics_at_saved_epoch = dict(        val_acc=float(saved_val_acc),        melanoma_sensitivity=float(saved_mel_sens),        bcc_sensitivity=float(saved_bcc_sens),    ),    provenance = dict(        notebook='06_swin_retrain.ipynb',        reason='recovery retrain; shipped checkpoint did not reproduce its stored metrics',        recipe='verbatim from src/models/swin*_5class.ipynb',        seed=SEED, gpu=GPU_NAME,        torch=torch.__version__, timm=timm.__version__,        data_root=str(LOCAL_ROOT), split_sizes=SPLIT_SIZES,        train_distribution=TRAIN_DIST, loss_weights=LOSS_WEIGHTS,        wall_clock_hours=float(train_wall/3600),        generated_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),    ),)torch.save(payload, save_path)print(f'saved {save_path}  ({save_path.stat().st_size/1024**2:.1f} MB)')print(f'  test acc  : {test_acc*100:.2f}%')print(f'  macro AUC : {macro_auc:.4f}')print(f'  MCC       : {mcc:.4f}')print(f'  saved epoch: {saved_epoch} of {epochs_run} run')

## 10. Save-then-verify — the check that was missing**This is the cell whose absence cost the original weights.**It reloads the file that was just written, rebuilds the model from scratch, loads thestate dict with `strict=True`, and asserts:1. every tensor in the reloaded state dict is bit-identical to the in-memory model;2. re-scoring the test set from the reloaded file reproduces `test_acc` exactly;3. the stored `test_acc` and the recomputed one agree to < 1e-9.If any assertion fires, **the saved file does not match the model that was justevaluated** and the checkpoint must not be used. The original `swin` checkpointwould have failed assertion 1.

In [ ]:
TOL = 1e-9in_memory = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}reloaded = torch.load(save_path, map_location='cpu', weights_only=False)assert list(reloaded['classes']) == CLASSES, 'class order changed on round-trip'verify_model = SwinSkinClassifier(    model_name=CFG['model_name'], num_classes=NUM_CLS,    dropout=CFG['dropout'], drop_path_rate=CFG['drop_path_rate'], pretrained=False)verify_model.load_state_dict(reloaded['model_state'], strict=True)print('1/3  strict=True load from disk: OK '      f'({len(reloaded["model_state"])} tensors)')mismatch = []for k, v in in_memory.items():    w = reloaded['model_state'][k].detach().cpu()    if v.shape != w.shape or not torch.equal(v, w):        mismatch.append(k)assert not mismatch, (    f'{len(mismatch)} tensors differ between the saved file and the evaluated '    f'model, e.g. {mismatch[:5]} -- THE CHECKPOINT IS NOT THE MODEL THAT WAS SCORED')print(f'2/3  all {len(in_memory)} tensors bit-identical to the evaluated model: OK')verify_model = verify_model.to(DEVICE).to(memory_format=torch.channels_last).eval()vp, vl_ = [], []with torch.no_grad():    for imgs, lbls in test_loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        vp.extend(verify_model(imgs).argmax(1).cpu().numpy())        vl_.extend(lbls.numpy())re_acc = accuracy_score(np.array(vl_), np.array(vp))assert abs(re_acc - test_acc) < TOL, (    f'reloaded model scores {re_acc*100:.4f}% but the evaluated model scored '    f'{test_acc*100:.4f}% -- REPRODUCTION FAILED')assert abs(reloaded['test_acc'] - re_acc) < TOLprint(f'3/3  reloaded model reproduces test_acc exactly: '      f'{re_acc*100:.4f}% == {test_acc*100:.4f}%: OK')VERIFY = dict(passed=True, n_tensors=len(in_memory),              test_acc_in_memory=float(test_acc),              test_acc_reloaded=float(re_acc),              stored_test_acc=float(reloaded['test_acc']))print('\nCHECKPOINT VERIFIED — safe to use for the paper.')del verify_modelif DEVICE.type == 'cuda':    torch.cuda.empty_cache()

## 11. `timing.json`, `results_*.json` and training curves

In [ ]:
timing = dict(    model='Swin-Tiny', timm_name=CFG['model_name'],    gpu=GPU_NAME,    gpu_count=1,    vram_gb=float(torch.cuda.get_device_properties(0).total_memory/1024**3)            if DEVICE.type == 'cuda' else None,    epochs_run=int(epochs_run), epochs_configured=int(CFG['num_epochs']),    wall_clock_seconds=float(train_wall),    wall_clock_hours=float(train_wall/3600),    mean_epoch_minutes=float(np.mean(epoch_seconds)/60),    batch_size=int(CFG['batch_size']),    amp=bool(CFG['use_amp']),    note='Measured on Kaggle, one T4. Replaces the unsupported "2xT4, 32 GB, ~6 h" '         'claim; the original runs were on Google Colab.',)with open(SAVE_DIR/'timing.json', 'w', encoding='utf-8') as f:    json.dump(timing, f, indent=2)results = dict(    meta=dict(notebook='06_swin_retrain.ipynb', model='Swin-Tiny',              seed=SEED, gpu=GPU_NAME,              torch=torch.__version__, timm=timm.__version__,              generated_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),              recipe='verbatim from the original notebook; see notebook header'),    config=CFG,    splits=SPLIT_SIZES, train_distribution=TRAIN_DIST,    loss_weights=LOSS_WEIGHTS,    test=dict(accuracy=float(test_acc), macro_auc=float(macro_auc), mcc=float(mcc),              balanced_accuracy=float(bal_acc), macro_f1=float(macro_f1),              n=int(len(all_labels))),    per_class=PER_CLASS,    confusion_matrix=cm.tolist(),    saved_epoch=int(saved_epoch),    metrics_at_saved_epoch=dict(val_acc=float(saved_val_acc),                                melanoma_sensitivity=float(saved_mel_sens),                                bcc_sensitivity=float(saved_bcc_sens)),    best_composite=float(best_composite),    history=history,    timing=timing,    verification=VERIFY,    original_checkpoint_for_comparison=dict(        stored_test_acc=0.853900, stored_macro_auc=0.9683,        note='The shipped checkpoint stored these but did not reproduce them; '             'it scored far lower even on its own training split.'),)with open(SAVE_DIR/'results_swin_v2.json', 'w', encoding='utf-8') as f:    json.dump(results, f, indent=2)print('wrote timing.json and results_swin_v2.json')ep = range(1, epochs_run + 1)fig, ax = plt.subplots(2, 3, figsize=(17, 9)); ax = ax.flatten()ax[0].plot(ep, history['train_loss'], label='train'); ax[0].plot(ep, history['val_loss'], label='val')ax[0].set_title('Loss'); ax[0].legend(); ax[0].set_xlabel('epoch')ax[1].plot(ep, [a*100 for a in history['train_acc']], label='train')ax[1].plot(ep, [a*100 for a in history['val_acc']], label='val')ax[1].set_title('Accuracy (%)'); ax[1].legend(); ax[1].set_xlabel('epoch')ax[2].plot(ep, [g*100 for g in history['gap']]); ax[2].axhline(5, ls='--', c='orange')ax[2].axhline(10, ls='--', c='red'); ax[2].set_title('train - val gap (pp)'); ax[2].set_xlabel('epoch')ax[3].plot(ep, [m*100 for m in history['melanoma_sensitivity']], label='melanoma')ax[3].plot(ep, [b*100 for b in history['bcc_sensitivity']], label='BCC')ax[3].axhline(95, ls='--', c='red', label='mel target')ax[3].set_title('Malignant sensitivity (%)'); ax[3].legend(); ax[3].set_xlabel('epoch')ax[4].plot(ep, history['composite'])ax[4].axvline(saved_epoch, ls='--', c='green', label=f'saved @ {saved_epoch}')ax[4].set_title('Composite (checkpoint criterion)'); ax[4].legend(); ax[4].set_xlabel('epoch')ax[5].plot(ep, [t/60 for t in epoch_seconds]); ax[5].set_title('Minutes per epoch')ax[5].set_xlabel('epoch')for a in ax: a.grid(alpha=0.3)plt.suptitle(f'Swin-Tiny retrain — test {test_acc*100:.2f}%, '             f'MCC {mcc:.4f}, {epochs_run} epochs, {train_wall/3600:.2f} h on {GPU_NAME}',             fontsize=12.5, fontweight='bold')plt.tight_layout()plt.savefig(SAVE_DIR/'curves_swin_v2.pdf', bbox_inches='tight')plt.close()print('wrote curves_swin_v2.pdf')print('\n' + '='*72)print(f'Swin-Tiny RETRAIN COMPLETE')print('='*72)print(f'  test accuracy : {test_acc*100:.2f}%   (shipped checkpoint claimed 85.39%)')print(f'  macro AUC     : {macro_auc:.4f}  (claimed 0.9683)')print(f'  MCC           : {mcc:.4f}')print(f'  epochs run    : {epochs_run} / {CFG["num_epochs"]}')print(f'  wall-clock    : {train_wall/3600:.2f} h on {GPU_NAME}')print(f'  verified      : {VERIFY["passed"]}')print()print('Download from /kaggle/working/ before the session expires:')print(f'  swin_tiny_5cls_skin_v2.pth   <- the checkpoint')print(f'  results_swin_v2.json, history.csv, timing.json')print(f'  per_class_test.csv, confusion_test.csv, curves_swin_v2.pdf')

## 12. Next steps1. Download `swin_tiny_5cls_skin_v2.pth` and upload it to the Kaggle checkpoint dataset.2. Repeat for the other two backbones (EfficientNet + ConvNeXt).   ConvNeXt uses Notebook 04, which additionally fixes its premature stop   (`freeze_epochs` 8->3, `patience` 5->10) — its original run ended at epoch 9 of 30.3. Once all three exist, re-run Notebook 1. Its reproduction gate will then check the   **new** stored metrics, and every table, CI and figure in the paper comes from that run.**Kaggle GPU budget:** roughly 9 h per week. This notebook plus one more fits in aweek; the third goes in the next.